In [1]:
# 1. Load metadata (i: dataset folder directory for LINCS, o: 3 metadata files loaded)
# 2. Inspect metadata (i: 3 metadata files, o: -)
# 3. Analyze Finferprint collision issues => handle issues appropriately
# 4. Load Dataset
# 5. Inspect dataset
# 6. preprocess the data

# Load Packages 

In [2]:
# Core
import os

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from pandarallel import pandarallel  # For making applying of a function faster

pandarallel.initialize(progress_bar=True)

# Show all columns
pd.set_option("display.max_columns", None)
# Disable internal RDKit logs

INFO: Pandarallel will run on 15 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [3]:
# scripts
from data_loading import load_metadata_txt, LINCSDataLoader
from inspect_fingerprints import get_fingerprint, analyze_fingerprint_collision, inspect_reason

In [4]:
# Utils
from pathlib import Path

# For gctx
from cmapPy.pandasGEXpress.parse import parse

# RDKit stuff
from rdkit import RDLogger

# Calculations

RDLogger.DisableLog("rdApp.*")

In [5]:
import warnings
from warnings import simplefilter

# Suppress specific FutureWarning from cmapPy/pandas interaction
warnings.filterwarnings("ignore", category=FutureWarning, module="cmapPy")
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

# Load Metadata

In [6]:
METADATA_EDITED_FOLDER = Path("./Metadata_edited/")
if not os.path.exists(METADATA_EDITED_FOLDER):
    os.mkdir(METADATA_EDITED_FOLDER)

In [7]:
# Path is user input
merged_LINCS_metadata_dir = Path("/Users/ani/Thesis/prnet_eval/dataset/metadata/LINCS")
save_dir_analysis_merged = merged_LINCS_metadata_dir / "Analysis"

In [8]:
comp_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "compoundinfo_beta.txt")
gene_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "geneinfo_beta.txt")
inst_info_merged = load_metadata_txt(merged_LINCS_metadata_dir / "instinfo_beta.txt")

# Inspect Metadata

In [9]:
inst_info_merged["pert_type"].unique()

array(['trt_sh', 'ctl_vector', 'trt_lig', 'trt_oe', 'trt_cp', 'trt_aby',
       'trt_xpr', 'ctl_x', 'ctl_vehicle', 'ctl_untrt', 'trt_si'],
      dtype=object)

In [10]:
controls = ["ctl_x", "ctl_vehicle", "ctl_untrt", "ctl_vector"] 

inst_info_merged_control = inst_info_merged[
    inst_info_merged["pert_type"].isin(controls)
]
inst_info_merged_control

,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_itime,pert_time_unit,cell_mfc_name,pert_mfc_id,det_plate,det_well,rna_plate,rna_well,count_mean,count_cv,qc_f_logp,qc_iqr,qc_slope,pert_id,sample_id,pert_type,cell_iname,qc_pass,dyn_range,inv_level_10,build_name,failure_mode,project_code,cmap_name
1,b10,,1.0,uL,1 uL,96.0,96 h,h,U2OS,TRCN0000072237,TAK004_U2OS_96H_X2_B10_DUO52HI53LO,D10,TAK004_U2OS_96H_X1,D10,67,18,5.7,14.98,67,TRCN0000072237,TAK004_U2OS_96H_X2_B10_DUO52HI53LO:D10,ctl_vector,U2OS,0,4.73906,1462,,inv_level_10,TAK,LACZ
10,b11,,20.0,uL,20 uL,120.0,120 h,h,VCAP,TRCN0000072261,ERG013_VCAP_120H_X1_B11,A12,ERG013_VCAP_120H_X1,A12,108,14,5.9,22.92,70,TRCN0000072261,ERG013_VCAP_120H_X1_B11:A12,ctl_vector,VCAP,0,3.3136,1511,,"inv_level_10,qc_iqr,dyn_range",ERG,LUCIFERASE
11,b12,,150.0,ng,150 ng,48.0,48 h,h,HEK293T,GFP,HSF043_HEK293T_48H_X1_B12,O04,HSF043_HEK293T_48H_X1,O04,91,15,6.5,11.23,63,GFP,HSF043_HEK293T_48H_X1_B12:O04,ctl_vector,HEK293T,1,6.44316,3344,,None,HSF,GFP
15,b11,,20.0,uL,20 uL,72.0,72 h,h,VCAP,TRCN0000072261,ERG013_VCAP_72H_X1_B11,K24,ERG013_VCAP_72H_X1,K24,36,20,7.3,12.11,67,TRCN0000072261,ERG013_VCAP_72H_X1_B11:K24,ctl_vector,VCAP,0,4.13253,2401,,"count_mean,dyn_range",ERG,LUCIFERASE
31,b12,,150.0,ng,150 ng,48.0,48 h,h,HEK293T,GFP,HSF039_HEK293T_48H_X1_B12,O01,HSF039_HEK293T_48H_X1,O01,74,17,6.6,8.72,58,GFP,HSF039_HEK293T_48H_X1_B12:O01,ctl_vector,HEK293T,1,20.3247,3130,,None,HSF,GFP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1279962,f2b4,,NaN,None,None,24.0,24 h,h,MCF7,DMSO,DOS033_MCF7_24H_X1_F2B4_DUO52HI53LO,E17,DOS033_MCF7_24H_X1,E17,79,29,7.8,12.27,68,DMSO,DOS033_MCF7_24H_X1_F2B4_DUO52HI53LO:E17,ctl_vehicle,MCF7,0,6.14767,1915,,count_cv,DOS,DMSO
1279968,f2b4,,NaN,None,None,24.0,24 h,h,PC3,DMSO,DOS034_PC3_24H_X3_F2B4_DUO52HI53LO,E17,DOS034_PC3_24H_X3,E17,91,29,7.3,5.07,39,DMSO,DOS034_PC3_24H_X3_F2B4_DUO52HI53LO:E17,ctl_vehicle,PC3,0,8.58678,2078,,"count_cv,qc_iqr,qc_slope",DOS,DMSO
1279990,f2b4,,NaN,None,None,96.0,96 h,h,HCC515,CMAP-000,KDC010_HCC515_96H_X1_F2B4_DUO52HI53LO,D23,KDC010_HCC515_96H_X1,D23,79,28,7.5,13.01,68,CMAP-000,KDC010_HCC515_96H_X1_F2B4_DUO52HI53LO:D23,ctl_untrt,HCC515,0,7.5125,1502.5,,"count_cv,inv_level_10",KDC,UnTrt
1280000,f2b5,,NaN,None,None,24.0,24 h,h,MDAMB231,DMSO,LJP002_MDAMB231_24H_X1_F2B5_DUO52HI53LO,D02,LJP002_MDAMB231_24H_X1,D02,108,15,8.1,12.11,66,DMSO,LJP002_MDAMB231_24H_X1_F2B5_DUO52HI53LO:D02,ctl_vehicle,MDAMB231,1,8.5669,2433,,None,LJP,DMSO


In [11]:
gene_info_merged 

,gene_id,gene_symbol,ensembl_id,gene_title,gene_type,src,feature_space
0,750,GAS8-AS1,ENSG00000221819,GAS8 antisense RNA 1,ncRNA,NCBI,inferred
1,6315,ATXN8OS,None,ATXN8 opposite strand lncRNA,ncRNA,NCBI,inferred
2,7503,XIST,ENSG00000229807,X inactive specific transcript,ncRNA,NCBI,inferred
3,8552,INE1,ENSG00000224975,inactivation escape 1,ncRNA,NCBI,inferred
4,9834,FAM30A,ENSG00000226777,family with sequence similarity 30 member A,ncRNA,NCBI,inferred
...,...,...,...,...,...,...,...
12323,100287932,TIMM23,ENSG00000265354,translocase of inner mitochondrial membrane 23,protein-coding,NCBI,best inferred
12324,100289678,ZNF783,ENSG00000204946,zinc finger family member 783,protein-coding,NCBI,best inferred
12325,100507436,MICA,ENSG00000204520,MHC class I polypeptide-related sequence A,protein-coding,NCBI,best inferred
12326,9142,TMEM257,ENSG00000221870,transmembrane protein 257,protein-coding,NCBI,best inferred


In [12]:
comp_duplicate_count_smiles = comp_info_merged['canonical_smiles'].value_counts()
comp_duplicate_count_alias = comp_info_merged['compound_aliases'].value_counts()


In [13]:
comp_duplicate_count_smiles[comp_duplicate_count_smiles > 1]

canonical_smiles
CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F)F)c(F)c2)ccn1                                                 96
CCN(CC)CCNC(=O)c1c(C)[nH]c(C=C2/C(=O)Nc3ccc(F)cc23)c1C                                                       80
CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F)F)cc2)ccn1                                                    66
CNC(=O)c1cccc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F)F)cc2)c1                                                    66
CC1(C)O[C@@H]2CO[C@@]3(COS(N)(=O)=O)OC(C)(C)O[C@H]3[C@@H]2O1                                                 66
                                                                                                             ..
CCN(C(=O)C=CC)c1ccccc1C                                                                                       2
NC(=O)C1(CCN(CCCC(=O)c2ccc(F)cc2)CC1)N1CCCCC1                                                                 2
CN(C)CCOc1ccc(cc1)C(=C(CCCl)c1ccccc1)c1ccccc1                                          

In [14]:
comp_duplicate_count_alias[comp_duplicate_count_alias > 1]

compound_aliases
sunitinib-malate          40
7-HYDROXYSTAUROSPORINE    18
quetiapine-fumarate       14
dactolisib                 8
M-3M3FBS                   5
                          ..
r(-)-apomorphine           2
TIPIFARNIB-P2              2
medroxyprogesterone        2
methylergometrine          2
2-PHENYLMELATONIN          2
Name: count, Length: 83, dtype: int64

In [15]:
gene_info_merged.head()

,gene_id,gene_symbol,ensembl_id,gene_title,gene_type,src,feature_space
0,750,GAS8-AS1,ENSG00000221819,GAS8 antisense RNA 1,ncRNA,NCBI,inferred
1,6315,ATXN8OS,None,ATXN8 opposite strand lncRNA,ncRNA,NCBI,inferred
2,7503,XIST,ENSG00000229807,X inactive specific transcript,ncRNA,NCBI,inferred
3,8552,INE1,ENSG00000224975,inactivation escape 1,ncRNA,NCBI,inferred
4,9834,FAM30A,ENSG00000226777,family with sequence similarity 30 member A,ncRNA,NCBI,inferred


In [16]:
# Subset just for compounds
inst_info_merged_comp = inst_info_merged[inst_info_merged["pert_type"] == "trt_cp"]

In [17]:
inst_info_merged_comp.head()

,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_itime,pert_time_unit,cell_mfc_name,pert_mfc_id,det_plate,det_well,rna_plate,rna_well,count_mean,count_cv,qc_f_logp,qc_iqr,qc_slope,pert_id,sample_id,pert_type,cell_iname,qc_pass,dyn_range,inv_level_10,build_name,failure_mode,project_code,cmap_name
4,f3b5,6.66,5.3300,uM,6.66 uM,24.0,24 h,h,A375,BRD-K79781870,DOS043_A375_24H_X1_F3B5_DUO52HI53LO,D17,DOS043_A375_24H_X1,D17,58,26,6.3,15.87,68,BRD-K79781870,DOS043_A375_24H_X1_F3B5_DUO52HI53LO:D17,trt_cp,A375,0,6.78867,1558,,"inv_level_10,qc_iqr",DOS,BRD-K79781870
7,f1b10,0.002,0.0015,uM,0.002 uM,6.0,6 h,h,A549,BRD-A61304759,HOG001_A549_6H_X4_F1B10,P11,HOG001_A549_6H_X4,P11,65,20,9.8,9.95,62,BRD-A61304759,HOG001_A549_6H_X4_F1B10:P11,trt_cp,A549,1,12.4715,2301,,None,HOG,tanespimycin
13,f1b10,0.02,0.0152,uM,0.02 uM,6.0,6 h,h,A549,BRD-K18190982,HOG002_A549_6H_X1_F1B10,G09,HOG002_A549_6H_X1,G09,46,21,8.6,10.63,64,BRD-K18190982,HOG002_A549_6H_X1_F1B10:G09,trt_cp,A549,1,8.83029,4110.5,,None,HOG,COT-10b
14,f1b10,0.002,0.0015,uM,0.002 uM,24.0,24 h,h,A549,BRD-A85860691,HOG001_A549_24H_X1_F1B10,J22,HOG001_A549_24H_X1,J22,71,18,7.7,9.74,59,BRD-A85860691,HOG001_A549_24H_X1_F1B10:J22,trt_cp,A549,1,11.5843,3023.5,,None,HOG,chaetocin
19,b10,0.002,0.0015,uM,0.002 uM,24.0,24 h,h,MCF7,BRD-K05649647,HOG002_MCF7_24H_X3_B10,O11,HOG002_MCF7_24H_X3,O11,60,18,6.1,15.2,67,BRD-K05649647,HOG002_MCF7_24H_X3_B10:O11,trt_cp,MCF7,0,3.01029,3802,,"qc_iqr,dyn_range",HOG,BRD-K05649647


# Fingerprint Collisions

In [18]:
comp_info_merged_path = METADATA_EDITED_FOLDER / "compounds_info_fingerprints.parquet"
if os.path.isfile(comp_info_merged_path):
    # Read Parquet instead of CSV
    comp_info_merged = pd.read_parquet(comp_info_merged_path)
else:
    comp_info_merged["Fingerprint_smiles"] = comp_info_merged[
        "canonical_smiles"
    ].parallel_apply(get_fingerprint)

    # Save to Parquet
    comp_info_merged.to_parquet(comp_info_merged_path, index=False)

In [19]:
collision_df_path = METADATA_EDITED_FOLDER / "collision_df.parquet"
if os.path.isfile(collision_df_path):
    # Read Parquet instead of CSV
    collision_df_merged = pd.read_parquet(collision_df_path)
else:
    results = []
    for fingerprint, group_df in comp_info_merged.groupby("Fingerprint_smiles"):
        if len(group_df) < 2:
            continue

        smiles_list = group_df["canonical_smiles"].to_list()
        reasons = analyze_fingerprint_collision(smiles_list)

        results.append(
            {
                "Fingerprint": fingerprint,
                "No. compounds": len(smiles_list),
                "Reasons": ", ".join(reasons),
                "Smiles": smiles_list,
                "cmap_name": group_df["cmap_name"].to_list(),
                "pert_id": group_df["pert_id"].to_list(),
            }
        )
    collision_df_merged = pd.DataFrame(results)

    # Save to Parquet
    collision_df_merged.to_parquet(collision_df_path, index=False)

In [20]:
collision_df_merged.head()

,Fingerprint,No. compounds,Reasons,Smiles,cmap_name,pert_id
0,0000100010000000000000000000000000000000000000...,2,True duplicates,"[[I+]1c2ccccc2-c2ccccc12, [I+]1c2ccccc2-c2cccc...","[diphenyleneiodonium, diphenyleneiodonium]","[BRD-K65814004, BRD-K65814004]"
1,0000101000000000000100000000000000000000000000...,3,True duplicates,"[Nc1nc(N)c2nc(-c3ccccc3)c(N)nc2n1, Nc1nc(N)c2n...","[triamterene, triamterene, triamterene]","[BRD-K92049597, BRD-K92049597, BRD-K92049597]"
2,0000101000000000000100000000000000000000000010...,16,True duplicates,"[Nc1ccncc1, Nc1ccncc1, Nc1ccncc1, Nc1ccncc1, N...","[dalfampridine, dalfampridine, dalfampridine, ...","[BRD-K22482860, BRD-K22482860, BRD-K22482860, ..."
3,0000101010000000000100000000000000000000000000...,6,True duplicates,"[Nc1nnc(c(N)n1)-c1cccc(Cl)c1Cl, Nc1nnc(c(N)n1)...","[lamotrigine, lamotrigine, lamotrigine, lamotr...","[BRD-K93460210, BRD-K93460210, BRD-K93460210, ..."
4,0000101010000000000100000000000000000000000000...,2,True duplicates,"[Clc1cccc(Nc2nnc(-c3ccccc3)c3ccccc23)c1, Clc1c...","[MY-5445, MY-5445]","[BRD-K90524085, BRD-K90524085]"


In [21]:
compounds_per_reasons = dict(
    sorted(
        {
            reas: (group["No. compounds"].sum(), len(group))
            for reas, group in collision_df_merged.groupby("Reasons")
        }.items(),
        key=lambda item: item[1],
        reverse=True,
    )
)
print(
    f"Total number of affected compounds: {np.array(list(compounds_per_reasons.values()))[:, 0].sum()}"
)
print(f"{100 * '-'}\n")
for key, val in compounds_per_reasons.items():
    print(f"{key:<140}: {val[0]:<5} compounds in {val[1]:<5} groups")

Total number of affected compounds: 27414
----------------------------------------------------------------------------------------------------

Stereoisomers (R/S conflict)                                                                                                                : 19714 compounds in 3813  groups
True duplicates                                                                                                                             : 4563  compounds in 975   groups
Defined vs Undefined Stereo                                                                                                                 : 754   compounds in 174   groups
Defined vs Undefined Stereo, Stereoisomers (R/S conflict)                                                                                   : 609   compounds in 73    groups
Compositional Isomer (Different Formula), Stereoisomers (R/S conflict)                                                                      : 516   compounds in

In [22]:
inspect_reason(collision_df_merged, "True duplicates", max_print=20, draw_mols=False)

#### True duplicates ####
No. fingerprints: 975
Cc1cccc(Nc2ccccc2C(O)=O)c1C: mefenamic-acid, BRD-K92778217
Cc1cccc(Nc2ccccc2C(O)=O)c1C: mefenamic-acid, BRD-K92778217
------------------------------
COc1cc(CNC(=O)CCCC/C=CC(C)C)ccc1O: capsaicin, BRD-K16336526
COc1cc(CNC(=O)CCCCC=CC(C)C)ccc1O: capsaicin, BRD-K50590187
------------------------------
CC(C)(C)NCC(O)COc1cccc2NC(=O)CCc12: carteolol, BRD-A42167015
CC(C)(C)NCC(O)COc1cccc2NC(=O)CCc12: carteolol, BRD-A42167015
------------------------------
Clc1cccc(c1)N1CCNCC1: mCPP, BRD-K75844781
Clc1cccc(c1)N1CCNCC1: mCPP, BRD-K75844781
Clc1cccc(c1)N1CCNCC1: mCPP, BRD-K75844781
Clc1cccc(c1)N1CCNCC1: mCPP, BRD-K75844781
------------------------------
OC(COc1cccc2ncccc12)CN1CCN(CC1)C(=O)C(c1ccccc1)c1ccccc1: dofequidar, BRD-A14941520
OC(COc1cccc2ncccc12)CN1CCN(CC1)C(=O)C(c1ccccc1)c1ccccc1: dofequidar, BRD-A14941520
OC(COc1cccc2ncccc12)CN1CCN(CC1)C(=O)C(c1ccccc1)c1ccccc1: dofequidar, BRD-A14941520
OC(COc1cccc2ncccc12)CN1CCN(CC1)C(=O)C(c1ccccc1)c1ccc

In [23]:
inspect_reason(collision_df_merged, "Error in the labelling", max_print=10, draw_mols=True)

#### Error in the labelling ####
Category not found.



In [24]:
for reason in collision_df_merged["Reasons"].unique().tolist():
    inspect_reason(collision_df_merged, reason, max_print=2, draw_mols=False) # set to true if you want drawings

#### True duplicates ####
No. fingerprints: 975
Cc1cccc(Nc2ccccc2C(O)=O)c1C: mefenamic-acid, BRD-K92778217
Cc1cccc(Nc2ccccc2C(O)=O)c1C: mefenamic-acid, BRD-K92778217
------------------------------
COc1cc(CNC(=O)CCCC/C=CC(C)C)ccc1O: capsaicin, BRD-K16336526
COc1cc(CNC(=O)CCCCC=CC(C)C)ccc1O: capsaicin, BRD-K50590187
------------------------------
----------------------------------------------------------------------------------------------------
#### Bioisosteres (Same Skeleton), Structural Isomers (Positional) ####
No. fingerprints: 35
COC(=O)C1=C(C)NC(C)=C(C1c1ccccc1[N+]([O-])=O)C(=O)OC: nifedipine, BRD-K96354014
COC(=O)C1=C(C)NC(C)=C(C1c1ccccc1[N+]([O-])=O)C(=O)OC: nifedipine, BRD-K96354014
COC(=O)C1=C(C)NC(C)=C(C1c1ccccc1[N+]([O-])=O)C(=O)OC: nifedipine, BRD-K96354014
COC(=O)C1=C(C)NC(C)=C(C1c1ccccc1[N+]([O-])=O)C(=O)OC: nifedipine, BRD-K96354014
COC(=O)C1=C(C)NC(C)=C(C1c1ccccc1[N+]([O-])=O)C(=O)OC: nifedipine, BRD-K96354014
COC(=O)C1C(C(=C(C)N=C1C)C(=O)OC)c2ccccc2[N+](=O)[O-]: nifed

# Loading GE

In [25]:
merged_LINCS_dataset_dir = Path("/Users/ani/Thesis/prnet_eval/dataset/data/LINCS")

In [26]:
save_dir_analysis = merged_LINCS_dataset_dir / "Analysis"
merged_LINCS_dataset_lvl3_path = (
    merged_LINCS_dataset_dir / "level3_beta_trt_cp_n1805898x12328.gctx"
)

In [27]:
dataloader_comp = LINCSDataLoader(
    gctx_path=merged_LINCS_dataset_lvl3_path,
    inst_info=inst_info_merged_comp,
    gene_info=gene_info_merged,
    gene_marker="landmark",
    comp_identifier="pert_id",
    cell_identifier="cell_iname",
    instance_identifier="sample_id",
)


Subsetting genes using feature_space: landmark
Initializing metadata...


In [28]:
dataloader_ctl = LINCSDataLoader(
    gctx_path=merged_LINCS_dataset_lvl3_path,
    inst_info=inst_info_merged_control,
    gene_info=gene_info_merged,
    comp_identifier="pert_id",
    cell_identifier="cell_iname",
    instance_identifier="sample_id",
)


Initializing metadata...


In [ ]:
# Assuming dataloader is your instantiated LINCSDataLoader
# (which already filtered for landmark genes via gene_marker="landmark" during init)

control = ["ctl_x", "ctl_vehicle", "ctl_untrt", "ctl_vector"] 
compound = ["trt_cp"]

inst_filters_ctl = {
    "cell_iname": ["MCF7", "A549"],  # Multiple cell lines
    "pert_time": 24,  # Single timepoint
    "pert_dose_unit": "uM",  # Specific dose unit
    "pert_id": "BRD-K79781870",
    "pert_type": control
}

inst_filters_comp = {
    "cell_iname": ["MCF7", "A549"],  # Multiple cell lines
    "pert_time": 24,  # Single timepoint
    "pert_dose_unit": "uM",  # Specific dose unit
    "pert_id": "BRD-K79781870",
    "pert_type": compound

}
gene_filters_comp = {"feature_space": "landmark"}
gene_filters_ctl = {"feature_space": ["landmark", "inferred", "best_inferred"]}


In [41]:
filtered_metadata_ctl, gene_metadata_ctl, expression_matrix_ctl = dataloader_ctl.get_gene_expression(inst_filters=inst_filters_ctl)

if expression_matrix_ctl is not None:
    print(
        f"Loaded {expression_matrix_ctl.shape[0]} profiles and {expression_matrix_ctl.shape[1]} genes."
    )

No instances found matching the provided filters.


ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
filtered_metadata_comp, gene_metadata_comp, expression_matrix_comp = dataloader_comp.get_gene_expression(
    inst_filters=inst_filters_comp, gene_filters=gene_filters_comp
)

if expression_matrix_comp is not None:
    print(
        f"Loaded {expression_matrix_comp.shape[0]} profiles and {expression_matrix_comp.shape[1]} genes."
    )

Loaded 2 profiles and 978 genes.


# Create Anndata Object

In [ ]:
"""
Steps to create anndata obj:
1. Load the dataset with LincsDataLoader
2. Create anndata object
    2.1 Obs
    create attributes: cov_drug_name, cov_drug_dose_name, control
    merge smiles via pert_id, since pert_id is not unique, here is how to proceed:
    - pert_id's occur multiple times because their salt form, batch or similar differ.
    - for smiles this is irrelevant, as they still have the same string since they are the same compund.
      (Please check this first in notebook)
    - deduplicate and ONLY map the information on the smiles strings

    -- Preprocessing ---
3. Divide the dataset into control perturbations and non-control
4. Each perturbed observation gets an unperturbed observation
5. Delete all unpaired observations left (show how many were deleted!)
"""

"\nSteps to create anndata obj:\n1. Load the dataset with LincsDataLoader\n2. Create anndata object\n    2.1 Obs\n    create attributes: cov_drug_name, cov_drug_dose_name, control\n    merge smiles via pert_id, since pert_id is not unique, here is how to proceed:\n    - pert_id's occur multiple times because their salt form, batch or similar differ.\n    - for smiles this is irrelevant, as they still have the same string since they are the same compund.\n      (Please check this first in notebook)\n    - deduplicate and ONLY map the information on the smiles strings\n\n    -- Preprocessing ---\n3. Divide the dataset into control perturbations and non-control\n4. Each perturbed observation gets an unperturbed observation\n5. Delete all unpaired observations left (show how many were deleted!)\n"

In [ ]:
duplicate_pert_id_df = comp_info_merged[comp_info_merged.duplicated(subset=["pert_id"], keep=False)]


In [ ]:
duplicate_pert_id_df

,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases,Fingerprint_smiles
605,BRD-K43002773,GDC-0068,AKT3,Akt inhibitor,C[C@@H]1C[C@H](C2=C1C(=NC=N2)N3CCN(CC3)C(=O)[C...,GRZXWCHAXNAUHY-NSISKUIASA-N,ipatasertib,1011101010000000000100000000000001100000000000...
606,BRD-K43002773,GDC-0068,AKT1,Akt inhibitor,C[C@@H]1C[C@H](C2=C1C(=NC=N2)N3CCN(CC3)C(=O)[C...,GRZXWCHAXNAUHY-NSISKUIASA-N,ipatasertib,1011101010000000000100000000000001100000000000...
607,BRD-K43002773,GDC-0068,AKT2,Akt inhibitor,C[C@@H]1C[C@H](C2=C1C(=NC=N2)N3CCN(CC3)C(=O)[C...,GRZXWCHAXNAUHY-NSISKUIASA-N,ipatasertib,1011101010000000000100000000000001100000000000...
608,BRD-A50998626,palomid-529,MTOR,Akt inhibitor,COc1ccc(COc2cc3oc(=O)c4cc(ccc4c3cc2OC)C(C)O)cc1,YEAHTLOYHVWAKW-UHFFFAOYSA-N,PALOMID-529,1011101000000000000000000000000001010000000000...
611,BRD-K45293975,7-hydroxystaurosporine,CDK4,CDK inhibitor,CN[C@@H]1C[C@H]2O[C@@](C)([C@@H]1OC)n1c3ccccc3...,PBCZSGKMGDDXIJ-KRUBCLEUSA-N,7-HYDROXYSTAUROSPORINE,1111101001000000000100000000000000010000000000...
...,...,...,...,...,...,...,...,...
39310,BRD-A81177136,KN-62,P2RX7,Calcium/calmodulin dependent protein kinase in...,CN(C(Cc1ccc(OS(=O)(=O)c2cccc3cnccc23)cc1)C(=O)...,RJVLFQBBRSMWHX-UHFFFAOYSA-N,None,1010101000000000000000000000000000000000000010...
39311,BRD-A81177136,KN-62,CAMK2A,Calcium/calmodulin dependent protein kinase in...,CN(C(Cc1ccc(OS(=O)(=O)c2cccc3cnccc23)cc1)C(=O)...,RJVLFQBBRSMWHX-UHFFFAOYSA-N,None,1010101000000000000000000000000000000000000010...
39314,BRD-K08542803,gambogic-acid,BCL2,Telomerase reverse transcriptase expression in...,CC(C)=CCC[C@@]1(C)Oc2c(CC=C(C)C)c3O[C@@]45[C@H...,GEZHEQNLKAOMCA-RRZNCOCZSA-N,None,1111100010000000000000101000000011000000000010...
39319,BRD-A62182663,YK-4279,DHX9,Binding of RNA helicase A to the transcription...,COc1ccc(cc1)C(=O)CC1(O)C(=O)Nc2c1c(Cl)ccc2Cl,HLXSCTYHLQHQDJ-UHFFFAOYSA-N,None,1111100010110000000000000000000000000000000000...


In [ ]:
# Check if the same pert_id's have the same smiles string
smiles_counts = duplicate_pert_id_df.groupby("pert_id")["canonical_smiles"].nunique()

clean_duplicates = smiles_counts[smiles_counts == 1]
conflicting_duplicates = smiles_counts[smiles_counts > 1]

print(f"The smiles strings are all equal: {clean_duplicates}")
print(f"Different smiles strings for same pert_id: {conflicting_duplicates}")

The smiles strings are all equal: pert_id
BRD-A00147595    1
BRD-A00520476    1
BRD-A00827783    1
BRD-A00938334    1
BRD-A00993607    1
                ..
BRD-M90449146    1
BRD-M98279124    1
BRD-U07805514    1
BRD-U86686840    1
BRD-U88459701    1
Name: canonical_smiles, Length: 1375, dtype: int64
Different smiles strings for same pert_id: Series([], Name: canonical_smiles, dtype: int64)


In [ ]:
comp_info_merged

,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases,Fingerprint_smiles
0,BRD-A08715367,L-theanine,None,None,CCNC(=O)CCC(N)C(O)=O,DATAGRPVKZEWHA-UHFFFAOYSA-N,l-theanine,1110000000000000000100001000000010000000000000...
1,BRD-A12237696,L-citrulline,None,None,NC(CCCNC(N)=O)C(O)=O,RHGKLRLOHDJJDR-UHFFFAOYSA-N,l-citrulline,1110000000000000000100001000000010000000000000...
2,BRD-A18795974,BRD-A18795974,None,None,CCCN(CCC)C1CCc2ccc(O)cc2C1,BLYMJBIZMIGWFK-UHFFFAOYSA-N,7-hydroxy-DPAT,1001100000001000001000000000000000000000000000...
3,BRD-A27924917,BRD-A27924917,None,None,NCC(O)(CS(O)(=O)=O)c1ccc(Cl)cc1,WBSMZVIMANOCNX-UHFFFAOYSA-N,2-hydroxysaclofen,1111100010000000000100001000000010000000000000...
4,BRD-A35931254,BRD-A35931254,None,None,CN1CCc2cccc-3c2C1Cc1ccc(O)c(O)c-31,VMWNQDUVQKEIOC-UHFFFAOYSA-N,r(-)-apomorphine,1001100000000000001000000000000000000000000000...
...,...,...,...,...,...,...,...,...
39316,BRD-K62685538,triptorelin,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)C[C@H](NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(...,VXKHXGOKWPXYNA-PGBVPBMZSA-N,None,1111111000000000010001000000000000000000000000...
39317,BRD-K62221994,T-98475,GNRHR,Gonadotropin releasing factor hormone receptor...,CC(C)OC(=O)c1cn(Cc2c(F)cccc2F)c3sc(c(CN(C)Cc4c...,RANJJVIMTOIWIN-UHFFFAOYSA-N,None,1110101010110000101000100000000000000000010000...
39318,BRD-K53397409,benzoic-acid,RAB9A,"Precursor for food preservatives, plasticizers...",OC(=O)c1ccccc1,WPYMKLBDIGXBTP-UHFFFAOYSA-N,None,0110100000000000000000001000000010000000000000...
39319,BRD-A62182663,YK-4279,DHX9,Binding of RNA helicase A to the transcription...,COc1ccc(cc1)C(=O)CC1(O)C(=O)Nc2c1c(Cl)ccc2Cl,HLXSCTYHLQHQDJ-UHFFFAOYSA-N,None,1111100010110000000000000000000000000000000000...


In [ ]:
""" 
The attributes needed by PRnet: 
obs: 'cell_id', 'det_plate', 'det_well', 'lincs_phase', 'pert_dose', 'pert_dose_unit', 'pert_id', 'pert_iname', 'pert_mfc_id', 'pert_time', 'pert_time_unit', 'pert_type', 'rna_plate', 'rna_well', 'condition', 'cell_type', 'dose', 'cov_drug_dose_name', 'cov_drug_name', 'control', 'canonical_smiles', 'SMILES', 'paired_control_index', 'cell_type_split_0', 'cell_type_split_1', 'cell_type_split_2', 'cell_type_split_3', 'cell_type_split_4', 'random_split_0', 'random_split_1', 'random_split_2', 'random_split_3', 'random_split_4', 'drug_split_0', 'drug_split_1', 'drug_split_2', 'drug_split_3', 'drug_split_4', 'cov_drug_dose_name_split_0', 'cov_drug_dose_name_split_1', 'cov_drug_dose_name_split_2', 'cov_drug_dose_name_split_3', 'cov_drug_dose_name_split_4'
var: 'pr_gene_title', 'pr_is_lm', 'pr_is_bing'
uns: 'cydata_pull', 'log1p'
"""

" \nThe attributes needed by PRnet: \nobs: 'cell_id', 'det_plate', 'det_well', 'lincs_phase', 'pert_dose', 'pert_dose_unit', 'pert_id', 'pert_iname', 'pert_mfc_id', 'pert_time', 'pert_time_unit', 'pert_type', 'rna_plate', 'rna_well', 'condition', 'cell_type', 'dose', 'cov_drug_dose_name', 'cov_drug_name', 'control', 'canonical_smiles', 'SMILES', 'paired_control_index', 'cell_type_split_0', 'cell_type_split_1', 'cell_type_split_2', 'cell_type_split_3', 'cell_type_split_4', 'random_split_0', 'random_split_1', 'random_split_2', 'random_split_3', 'random_split_4', 'drug_split_0', 'drug_split_1', 'drug_split_2', 'drug_split_3', 'drug_split_4', 'cov_drug_dose_name_split_0', 'cov_drug_dose_name_split_1', 'cov_drug_dose_name_split_2', 'cov_drug_dose_name_split_3', 'cov_drug_dose_name_split_4'\nvar: 'pr_gene_title', 'pr_is_lm', 'pr_is_bing'\nuns: 'cydata_pull', 'log1p'\n"

In [ ]:
lincs_adata = dataloader.create_anndata(False, comp_info_merged, inst_filters, gene_filters)
lincs_adata.obs

NameError: name 'dataloader' is not defined

In [ ]:
controls = ["ctl_x", "ctl_vehicle", "ctl_untrt", "ctl_vector"] 

inst_info_merged_control = inst_info_merged[
    inst_info_merged["pert_type"].isin(controls)
]
inst_info_merged_control

,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_itime,pert_time_unit,cell_mfc_name,pert_mfc_id,det_plate,det_well,rna_plate,rna_well,count_mean,count_cv,qc_f_logp,qc_iqr,qc_slope,pert_id,sample_id,pert_type,cell_iname,qc_pass,dyn_range,inv_level_10,build_name,failure_mode,project_code,cmap_name
1,b10,,1.0,uL,1 uL,96.0,96 h,h,U2OS,TRCN0000072237,TAK004_U2OS_96H_X2_B10_DUO52HI53LO,D10,TAK004_U2OS_96H_X1,D10,67,18,5.7,14.98,67,TRCN0000072237,TAK004_U2OS_96H_X2_B10_DUO52HI53LO:D10,ctl_vector,U2OS,0,4.73906,1462,,inv_level_10,TAK,LACZ
10,b11,,20.0,uL,20 uL,120.0,120 h,h,VCAP,TRCN0000072261,ERG013_VCAP_120H_X1_B11,A12,ERG013_VCAP_120H_X1,A12,108,14,5.9,22.92,70,TRCN0000072261,ERG013_VCAP_120H_X1_B11:A12,ctl_vector,VCAP,0,3.3136,1511,,"inv_level_10,qc_iqr,dyn_range",ERG,LUCIFERASE
11,b12,,150.0,ng,150 ng,48.0,48 h,h,HEK293T,GFP,HSF043_HEK293T_48H_X1_B12,O04,HSF043_HEK293T_48H_X1,O04,91,15,6.5,11.23,63,GFP,HSF043_HEK293T_48H_X1_B12:O04,ctl_vector,HEK293T,1,6.44316,3344,,None,HSF,GFP
15,b11,,20.0,uL,20 uL,72.0,72 h,h,VCAP,TRCN0000072261,ERG013_VCAP_72H_X1_B11,K24,ERG013_VCAP_72H_X1,K24,36,20,7.3,12.11,67,TRCN0000072261,ERG013_VCAP_72H_X1_B11:K24,ctl_vector,VCAP,0,4.13253,2401,,"count_mean,dyn_range",ERG,LUCIFERASE
31,b12,,150.0,ng,150 ng,48.0,48 h,h,HEK293T,GFP,HSF039_HEK293T_48H_X1_B12,O01,HSF039_HEK293T_48H_X1,O01,74,17,6.6,8.72,58,GFP,HSF039_HEK293T_48H_X1_B12:O01,ctl_vector,HEK293T,1,20.3247,3130,,None,HSF,GFP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1279962,f2b4,,NaN,None,None,24.0,24 h,h,MCF7,DMSO,DOS033_MCF7_24H_X1_F2B4_DUO52HI53LO,E17,DOS033_MCF7_24H_X1,E17,79,29,7.8,12.27,68,DMSO,DOS033_MCF7_24H_X1_F2B4_DUO52HI53LO:E17,ctl_vehicle,MCF7,0,6.14767,1915,,count_cv,DOS,DMSO
1279968,f2b4,,NaN,None,None,24.0,24 h,h,PC3,DMSO,DOS034_PC3_24H_X3_F2B4_DUO52HI53LO,E17,DOS034_PC3_24H_X3,E17,91,29,7.3,5.07,39,DMSO,DOS034_PC3_24H_X3_F2B4_DUO52HI53LO:E17,ctl_vehicle,PC3,0,8.58678,2078,,"count_cv,qc_iqr,qc_slope",DOS,DMSO
1279990,f2b4,,NaN,None,None,96.0,96 h,h,HCC515,CMAP-000,KDC010_HCC515_96H_X1_F2B4_DUO52HI53LO,D23,KDC010_HCC515_96H_X1,D23,79,28,7.5,13.01,68,CMAP-000,KDC010_HCC515_96H_X1_F2B4_DUO52HI53LO:D23,ctl_untrt,HCC515,0,7.5125,1502.5,,"count_cv,inv_level_10",KDC,UnTrt
1280000,f2b5,,NaN,None,None,24.0,24 h,h,MDAMB231,DMSO,LJP002_MDAMB231_24H_X1_F2B5_DUO52HI53LO,D02,LJP002_MDAMB231_24H_X1,D02,108,15,8.1,12.11,66,DMSO,LJP002_MDAMB231_24H_X1_F2B5_DUO52HI53LO:D02,ctl_vehicle,MDAMB231,1,8.5669,2433,,None,LJP,DMSO


#### Add Drug, cov_drug_name and cov_drug_dose_name 

In [ ]:
lincs_adata_obs = lincs_adata.obs

lincs_adata_obs['Drug'] = lincs_adata_obs.index
lincs_adata_obs['cov_drug_name'] = lincs_adata_obs['cell_iname'].astype(str)+ '_' + lincs_adata_obs['Drug'].astype(str)
lincs_adata_obs['cov_drug_dose_name'] = lincs_adata_obs['cell_iname'].astype(str)+ '_' + lincs_adata_obs['Drug'].astype(str)+ '_' + lincs_adata_obs['pert_dose'].astype(str)

lincs_adata_obs

,bead_batch,nearest_dose,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_itime,pert_time_unit,cell_mfc_name,pert_mfc_id,det_plate,det_well,rna_plate,rna_well,count_mean,count_cv,qc_f_logp,qc_iqr,qc_slope,pert_id,pert_type,cell_iname,qc_pass,dyn_range,inv_level_10,build_name,failure_mode,project_code,cmap_name,canonical_smiles,Drug,cov_drug_name,cov_drug_dose_name
sample_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
DOS043_A549_24H_X2_F3B5_DUO52HI53LO:D17,f3b5,6.66,5.33,uM,6.66 uM,24.0,24 h,h,A549,BRD-K79781870,DOS043_A549_24H_X2_F3B5_DUO52HI53LO,D17,DOS043_A549_24H_X2,D17,82,23,8.6,14.4,69,BRD-K79781870,trt_cp,A549,0,6.72353,1143,,inv_level_10,DOS,BRD-K79781870,CN1CCN(CC1)C(=O)c1cc2CN([C@H](CCO)c2c(n1)-c1cc...,DOS043_A549_24H_X2_F3B5_DUO52HI53LO:D17,A549_DOS043_A549_24H_X2_F3B5_DUO52HI53LO:D17,A549_DOS043_A549_24H_X2_F3B5_DUO52HI53LO:D17_5.33
DOS043_A549_24H_X3_F3B5_DUO52HI53LO:D17,f3b5,6.66,5.33,uM,6.66 uM,24.0,24 h,h,A549,BRD-K79781870,DOS043_A549_24H_X3_F3B5_DUO52HI53LO,D17,DOS043_A549_24H_X3,D17,74,24,7.9,15.91,71,BRD-K79781870,trt_cp,A549,0,4.39444,791,,"inv_level_10,qc_iqr,dyn_range",DOS,BRD-K79781870,CN1CCN(CC1)C(=O)c1cc2CN([C@H](CCO)c2c(n1)-c1cc...,DOS043_A549_24H_X3_F3B5_DUO52HI53LO:D17,A549_DOS043_A549_24H_X3_F3B5_DUO52HI53LO:D17,A549_DOS043_A549_24H_X3_F3B5_DUO52HI53LO:D17_5.33


#### Concat Control and Test Data

In [ ]:
adata_obs_all = pd.concat([adata_control.obs, adata_test.obs], ignore_index=True)
adata_obs_all.index = adata_obs_all.index.astype(str)
adata_X_all = np.concatenate((adata_control.X, adata_test.X),axis=0)

adata = ad.AnnData(adata_X_all,obs=adata_obs_all,var=lincs_adata.var)
adata

In [ ]:
lincs_obs_list_prnet = ['cell_id', 'det_plate', 'det_well', 'lincs_phase', 'pert_dose', 'pert_dose_unit', 'pert_id', 'pert_iname', 'pert_mfc_id', 'pert_time', 'pert_time_unit', 'pert_type', 'rna_plate', 'rna_well', 'condition', 'cell_type', 'dose', 'cov_drug_dose_name', 'cov_drug_name', 'control', 'canonical_smiles', 'SMILES', 'paired_control_index', 'cell_type_split_0', 'cell_type_split_1', 'cell_type_split_2', 'cell_type_split_3', 'cell_type_split_4', 'random_split_0', 'random_split_1', 'random_split_2', 'random_split_3', 'random_split_4', 'drug_split_0', 'drug_split_1', 'drug_split_2', 'drug_split_3', 'drug_split_4', 'cov_drug_dose_name_split_0', 'cov_drug_dose_name_split_1', 'cov_drug_dose_name_split_2', 'cov_drug_dose_name_split_3', 'cov_drug_dose_name_split_4']
lincs_adata.obs

set_a = set(lincs_obs_list_prnet)
set_b = set(lincs_adata.obs)

shared = set_a.intersection(set_b) 

all_differences = set_a.symmetric_difference(set_b)  

print(f"Equal in both: {list(shared)}")
print(f"Not equal (unique to either list): {list(all_differences)}")

only_in_a = set_a.difference(set_b)  

only_in_b = set_b.difference(set_a)  

print(f"Only in first list: {list(only_in_a)}")
print(f"Only in second list: {list(only_in_b)}")

Equal in both: ['rna_well', 'pert_time_unit', 'pert_dose', 'pert_mfc_id', 'det_well', 'pert_type', 'pert_time', 'det_plate', 'rna_plate', 'pert_dose_unit', 'pert_id']
Not equal (unique to either list): ['condition', 'random_split_3', 'drug_split_0', 'qc_iqr', 'pert_iname', 'control', 'cov_drug_dose_name_split_2', 'cov_drug_dose_name', 'dose', 'qc_f_logp', 'random_split_1', 'cell_type_split_0', 'cell_mfc_name', 'pert_itime', 'nearest_dose', 'qc_slope', 'drug_split_1', 'random_split_4', 'cov_drug_dose_name_split_1', 'bead_batch', 'cov_drug_dose_name_split_0', 'count_mean', 'pert_idose', 'failure_mode', 'cmap_name', 'lincs_phase', 'drug_split_2', 'dyn_range', 'cov_drug_name', 'random_split_2', 'cell_type_split_3', 'cell_iname', 'inv_level_10', 'cov_drug_dose_name_split_3', 'build_name', 'project_code', 'cell_type_split_1', 'random_split_0', 'cell_type_split_4', 'qc_pass', 'canonical_smiles', 'count_cv', 'cell_type', 'drug_split_3', 'cell_id', 'SMILES', 'drug_split_4', 'paired_control_inde

In [ ]:
"""
Add to obs: condition, control, pert_iname, cov_drug_dose_name, dose, canonical_smiles, cell_type, 
            cell_id, SMILES, lincs_phase, paired_control_index, cov_drug_name
remove from obs: build_name, inv_level_10, pert_idose, nearest_dose, project_code, qc_slope, failure_mode, cmap_name,
                 qc_f_logp, pert_itime, cell_iname, count_mean, dyn_range, bead_batch, qc_iqr, qc_pass, cell_mfc_name,
                 count_cv
"""